In [ ]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like transformers can change and make your code non-runnable :)

!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/env_setup.py
import env_setup
env_setup.install_env()

Download and install the Project repo:

In [ ]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
BRANCH_NAME = "main"

!git clone -b {BRANCH_NAME} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"

os.chdir("hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

### Load the model

In [ ]:
# Core Hypencoder model for outputing dense vector representations
from hypencoder_cb.modeling.hypencoder import Hypencoder, HypencoderDualEncoder, TextEncoder
from transformers import AutoTokenizer

hidden_layers = 2 # possible values 2,4,6,8

# models from the original hypencoder paper
# model_name = f"jfkback/hypencoder.{hidden_layers}_layer"


# models trained by me - based on SapBert, trained on BC5CDR-MeSH2015
model_name = f"Stevenf232/hypencoder_{hidden_layers}layer_SapBERT_context"


dual_encoder = HypencoderDualEncoder.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


query_encoder: Hypencoder = dual_encoder.query_encoder
passage_encoder: TextEncoder = dual_encoder.passage_encoder

### Move the model to the GPU

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

# Move the model to the GPU
passage_encoder.to(device)
query_encoder.to(device)

### Load datasets, pre-process, and tokenise

In [ ]:
from datasets import load_dataset

# there are all "positive" pairs
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")

In [ ]:
from datasets import concatenate_datasets

# use only test split
data = dataset['test']

# Update the variables to use the combined data
mention_names = data['mention']
mention_texts = data['mention_text']
entity_names = data['entity']
definitions = data['definition']

print(f"Combined dataset size: {len(data)}")
print(data[0])

## Processing the dataset:
### RQ2 - choose a context window (0, 64, 128, 256, 512) to test how well the model does inference on input with that context length.
Default used is 128

In [ ]:
context_window = 128

In [ ]:
# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window

# reduce the mention text and definition
window = [get_mt_window(mention,text, context_window) for mention, text in zip(mention_names, mention_texts)]
short_def = [text[:context_window] for text in definitions]

In [ ]:
# convert from type "datasets" to python list
queries = list(zip(mention_names, window))

passages = list(set(zip(entity_names, short_def))) # remove duplicate entities

# the output of the tokenizer contains 3 fields:
# input_ids, token_type_ids, and attention_mask
# all contain a tensor in the shape (number of queries, max number of tokens)

query_inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length = 512)
passage_inputs = tokenizer(passages, return_tensors="pt", padding=True, truncation=True, max_length = 512)


# Passage Encodings


In [ ]:
from tqdm import tqdm
import torch
from torch.amp import autocast

def batch_encode_passages(encoder ,passages):
  batch_size=256
  entity_name_features = []

  num_passages = passages["input_ids"].shape[0]

  with torch.no_grad(): # Disable gradient calculation (saves tons of memory)
    for i in tqdm(range(0, num_passages, batch_size), desc="Extracting features"):

        # extract entity features
        # Autocast does the math in fp16 where possible (default is fp32)
        with autocast("cuda"):
          features = encoder(
              input_ids=passages["input_ids"][i:i + batch_size].to(device),
              attention_mask=passages["attention_mask"][i:i + batch_size].to(device)
            ).representation

          entity_name_features.append(features.detach().cpu()) # Detach and move to CPU to save VRAM/RAM


  features_tensor = torch.cat(entity_name_features, dim=0)

  return features_tensor

In [ ]:
passage_embeddings = batch_encode_passages(passage_encoder, passage_inputs)

##Now, we create the q-nets.

For each q-net, we feed through it all the passages the calculate the similarity.

But the q-nets are created in batches, and every batch is represented as a single object `NoTorchSequential`. (Check out the `RepeatedDenseBlockConverter` class in q_net.py for more info)

This object expects an input in the shape (N, M, H):

* N = number of queries (mentions)

* M = number of passages (entities)

* H = Hidden dimension (e.g., 768 for BERT)



---



The passage embeddings have the shape (M, H) so we must create an additional dimension of size N.

This will be done like so:
`passages_batch = passages.unsqueeze(0).expand(num_queries, -1, -1)`

* `.unsqueeze()` adds a new dimension (in our case at location 0)

* `.expand()` "expands" that new dimension to be size "num_queries"

* `.expand()` creates a view, so it costs almost 0 memory! (compared to .repeat() which changes the tensor)

# Q-nets take **a lot** of memory.

Instead of creating all of them and then doing the similarity calculation, we will create batches and calculate similarities for just those q-nets, then discard those q-nets and move on to the next batch.

In [ ]:
def batch_encode_queries(encoder, queries, passage_embeddings):
  batch_size = 32
  similarity_scores = []

  num_queries = queries["input_ids"].shape[0]

  with torch.no_grad():
    for i in tqdm(range(0, num_queries, batch_size), desc="Creating q-nets and calculating similarity scores"):

        # create q-nets
        with autocast("cuda"):
          q_nets = encoder(
              input_ids=queries["input_ids"][i:i + batch_size].to(device),
              attention_mask=queries["attention_mask"][i:i + batch_size].to(device)
            ).representation


        passages_gpu = passage_embeddings.to(device)

        # Note: we use q_nets.num_queries (our repo's noTorch equivalent of q_nets.shape[0]) instead of batch_size
        # because the total number might not be divisible by batch_size so the last batch might be smaller than the actual batch size
        passages_batch = passages_gpu.unsqueeze(0).expand(q_nets.num_queries, -1, -1)

        # calculate similarity
        batch_scores = q_nets(passages_batch)
        similarity_scores.append(batch_scores.detach().cpu())


  scores_tensor = torch.cat(similarity_scores, dim=0)
  return scores_tensor


In [ ]:
similarity_scores = batch_encode_queries(query_encoder, query_inputs, passage_embeddings)

In [ ]:
# sanity check
similarity_scores

# Evaluation

In [ ]:
import numpy as np
import torch

# Create a reference list of unique entities that matches the order of passage_embeddings
entities_list = [{'entity': e, 'definition': d} for e, d in passages]

def evaluate(test_data, similarity_scores, k):
    """
    Modified evaluation function based on evaluate_at_k logic.
    Adapts tensor scores to the hit-based reporting style.
    """
    correct_at_1_count = 0
    correct_in_top_k_count = 0
    mrr_sum = 0.0

    # similarity_scores shape: (num_queries, num_passages, 1)
    scores = similarity_scores.squeeze(-1)
    num_queries = scores.shape[0]

    print(f"Evaluating at k={k}")

    for i in range(num_queries):
        # Get scores and find top k indices
        query_scores = scores[i]
        top_k_val, top_k_idx = torch.topk(query_scores, k=min(k, len(query_scores)))

        correct_entity_name = test_data['entity'][i]

        found_in_top_k = False
        rank = 0

        # Check top-k hits
        for j in range(len(top_k_idx)):
            current_rank = j + 1
            idx = top_k_idx[j].item()
            matched_entity = entities_list[idx]['entity']

            if matched_entity == correct_entity_name:
                found_in_top_k = True
                rank = current_rank
                if rank == 1:
                    correct_at_1_count += 1
                correct_in_top_k_count += 1
                mrr_sum += 1.0 / rank
                break

        # Logging
        print(f"--- Query {i+1} ---")
        print(f"Mention: {test_data['mention'][i]}")
        print(f"Correct Entity: {correct_entity_name}")
        print(f"Context: {window[i]}")

        if found_in_top_k:
            print(f"Correct entity found at rank: {rank}")
        else:
            print(f"Correct entity NOT found in top {k} results.")
            print(f"Top {k} (wrong) matches:")
            for j in range(len(top_k_idx)):
                idx = top_k_idx[j].item()
                print(f" Rank {j+1}: Entity: {entities_list[idx]['entity']}: Definition: {entities_list[idx]['definition'].strip()}")
        print("\n")

    accuracy_at_1 = correct_at_1_count / num_queries
    accuracy_at_k = correct_in_top_k_count / num_queries
    mrr = mrr_sum / num_queries

    print(f"Total queries: {num_queries}")
    print(f"Accuracy@1: {accuracy_at_1:.4f}")
    print(f"Accuracy@{k}: {accuracy_at_k:.4f}")
    print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")

    return accuracy_at_1, accuracy_at_k, mrr

### Save results to file

In [ ]:
import contextlib
from pathlib import Path

k = 5 # accuracy@k

result_dir = "results"

# for hidden layer experiments
# file_name = f"{hidden_layers}layers_@{k}.txt"

# for context window experiments
file_name = f"Hypencoder_{context_window}Context.txt"


path = os.path.join(result_dir, file_name)

Path(result_dir).mkdir(parents=True, exist_ok=True)

with open(path, 'w') as f:
    with contextlib.redirect_stdout(f):
        evaluate(data, similarity_scores, k=k)